# Python Data Types — Coding Challenges

Welcome to the coding-challenge notebook. It contains **10 hands-on problems** that exercise the built-in data types from Sessions 4 and 5 — lists, strings, sets, and dictionaries — in the concise, assertion-checked style used in real skill screenings. Each challenge is a short paragraph telling you *what* to build; the cell below it contains a reference solution with a `print('ok')` at the end. Every solution cell finishes with an `assert`, so you can check yourself instantly: pass the assert, get the `ok`, move on.

The 10 challenges form a mini-catalogue of the most reusable data-type patterns:

1. **Grouping** — bundling items under a key with `dict.setdefault` (Challenge 1).
2. **Set intersection** — common elements across several collections with `&` (Challenge 2).
3. **Inversion** — re-keying a dictionary with `setdefault` on the other side (Challenge 3).
4. **Frequency** — the `Counter` pattern for "k most frequent" (Challenges 4 and 6).
5. **Merging with defaults** — combining dicts with `get(key, 0)` (Challenge 5).
6. **Chunking** — slicing any sequence into fixed-size pieces (Challenge 7).
7. **Symmetric difference** — set difference composed in both directions (Challenge 8).
8. **Multi-key sorting** — `sorted(..., key=...)` with tuple keys for stable, mixed-direction ordering (Challenge 9).
9. **Nested building** — `setdefault` chains that grow groups-of-groups (Challenge 10).

**How to use this notebook:** read the challenge, try to write the transformation yourself in a scratch cell, then run the given cell to see whether your output would pass the same assertions. If a challenge took longer than 10 minutes, mark it for a second pass — the closing card discusses realistic screening pace.

There is no artificial restriction on helpers: `collections.Counter`, `set` operators, dictionary methods, and comprehensions are all fair game. The setup cell imports `numpy` and `pandas` and seeds `random` for reproducibility — the challenges themselves are pure Python data-type work, so those imports just keep the environment consistent. Work through them in order; each builds a small amount of muscle that the next one reuses.


## Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

pandas 3.0.5


### Challenge 1 — Group words by first letter with `setdefault`

The task: turn a flat list of words into a dictionary whose keys are **first letters** and whose values are the lists of words starting with that letter.

```python
words = ['apple', 'avocado', 'banana']
# expected -> {'a': ['apple', 'avocado'], 'b': ['banana']}
```

The star of this pattern is **`dict.setdefault(key, default)`**: it returns the value for `key`, inserting `default` into the dict if `key` is not present yet. That lets you write the whole "is the bucket there? if not create it, then append" dance in one line:

```python
out.setdefault(w[0], []).append(w)
```

Walk through it for `'apple'`: `w[0]` is `'a'`; `setdefault('a', [])` returns the (newly created) list for `'a'`; `.append('apple')` pushes the word in. The second apple-like word `'avocado'` finds the `'a'` bucket already present and simply appends.

- `w[0]` is the character at index 0 of the word — the group key.
- Without `setdefault` you would need `if w[0] not in out: out[w[0]] = []; out[w[0]].append(w)` — three lines instead of one.
- The order of the words inside each bucket follows the input order; the dict itself keeps insertion order (Python 3.7+).
- The assert `out['a'] == ['apple', 'avocado']` verifies both the key grouping and within-bucket ordering.


## Challenge 1

Group a list of words by their first letter into a dict.

In [2]:
words = ['apple', 'avocado', 'banana']
out = {}
for w in words:
    out.setdefault(w[0], []).append(w)
assert out['a'] == ['apple', 'avocado']
print('ok')

ok


### Challenge 2 — Intersection of three lists with sets

The task: find the elements that appear in **all three** lists. The classic tool is a set intersection, because `&` keeps exactly the items present in every operand.

```python
a, b, c = [1, 2, 3, 4], [2, 3, 5], [3, 2]
# set(a) & set(b) & set(c) == {2, 3}
```

- `set(a)` removes duplicates and gives you O(1) membership.
- `a & b` keeps elements in both; chaining `& c` additionally requires membership in the third — the result is the common core of all three.
- Because sets are unordered, the intersection loses the original order. Here order does not matter: the assert compares against the set `{2, 3}`.
- `list(set(a) & set(b) & set(c))` converts back to a list if the problem ever demands a list output; add `sorted(...)` when order is specified.
- Non-set alternative: `[x for x in a if x in b and x in c]` — but the set version is shorter, faster on large inputs, and self-documenting.


## Challenge 2

Find the intersection of three lists.

In [3]:
a, b, c = [1, 2, 3, 4], [2, 3, 5], [3, 2]
common = set(a) & set(b) & set(c)
assert common == {2, 3}
print('ok')

ok


### Challenge 3 — Invert a dictionary (values to keys)

The task: take a dict whose values are **lists**, and build the inverse mapping — each individual value becomes a *key*, and the collection of original keys that had it becomes the new value.

```python
d = {'a': [1, 2], 'b': [2]}
# expected -> inv[2] == ['a', 'b']
```

The pattern is `setdefault` on the *inverted* side:

```python
for k, vs in d.items():
    for v in vs:
        inv.setdefault(v, []).append(k)
```

Walk through it: `d.items()` yields `('a', [1, 2])` then `('b', [2])`. The inner loop visits `1` and `2` for key `'a'` (`inv[1] = ['a']`, `inv[2] = ['a']`), then visits `2` for key `'b'` — and since `2` already exists, `'b'` is appended: `inv[2] == ['a', 'b']`.

- The transform is **value → key**: whatever was a value in the input becomes a key in the output.
- This is the *two-level swap*: once with `setdefault` for grouping on the new key, exactly like Challenge 1.
- The assert `inv[2] == ['a', 'b']` is the signature of the pattern — one value shared by multiple keys lands in that key's bucket.


## Challenge 3

Invert a dict that maps key -> list of values.

In [4]:
d = {'a': [1, 2], 'b': [2]}
inv = {}
for k, vs in d.items():
    for v in vs:
        inv.setdefault(v, []).append(k)
assert inv[2] == ['a', 'b']
print('ok')

ok


### Challenge 4 — The k most frequent elements with `Counter`

The task: return the **k elements** that occur most often in a list, in descending frequency order. `collections.Counter` is built for exactly this.

```python
from collections import Counter
def top_k(values, k):
    return [v for v, _ in Counter(values).most_common(k)]
```

- `Counter(values)` counts every element: it is the idiomatic "frequency dictionary".
- `most_common(k)` returns the `k` most frequent `(element, count)` pairs, already ordered best-first; ties keep first-seen order.
- The comprehension `[v for v, _ in ...]` keeps only the elements, discarding the counts.
- `top_k([1, 1, 2, 2, 2, 3], 2)` → `most_common(2)` gives `[(2, 3), (1, 2)]` → result `[2, 1]` — the assert checks exactly that.
- Hand-rolled alternative (no imports): build `d[x] = d.get(x, 0) + 1`, sort items by count descending, take the first `k`. The `Counter` version is the Pythonic standard — know it well.


## Challenge 4

Return the k most frequent elements in a list.

In [5]:
from collections import Counter
def top_k(values, k):
    return [v for v, _ in Counter(values).most_common(k)]

assert top_k([1, 1, 2, 2, 2, 3], 2) == [2, 1]
print('ok')

ok


### Challenge 5 — Merge two dicts, summing shared keys

The task: combine two dictionaries so that keys present in **both** keep the **sum** of their two values, while keys present in only one keep their value.

```python
a, b = {'x': 1, 'y': 2}, {'y': 3, 'z': 4}
# expected -> {'x': 1, 'y': 5, 'z': 4}
```

The one-liner uses the dict-union operator `|` to enumerate every key exactly once, plus `get` with a default to read safely:

```python
merged = {k: a.get(k, 0) + b.get(k, 0) for k in a | b}
```

- `a | b` merges the key sets of both dicts (Python 3.9+; it yields the union of keys — the one key appearing... plus the shared `'y'` exactly once).
- `a.get(k, 0)` returns `a`'s value or `0` when the key is not in `a` — so a missing side contributes nothing to the sum.
- For `'y'`: `a.get('y',0) + b.get('y',0) == 2 + 3 == 5` — the assert.
- Older-Python alternative: start with a copy of one dict, then add the other's values onto shared keys.
- This is the "merge with aggregation" pattern — the basis of many counting/feature-engineering joins.


## Challenge 5

Merge two dicts summing values for shared keys.

In [6]:
a, b = {'x': 1, 'y': 2}, {'y': 3, 'z': 4}
merged = {k: a.get(k, 0) + b.get(k, 0) for k in a | b}
assert merged['y'] == 5
print('ok')

ok


### Challenge 6 — Permutation check with `Counter`

The task: decide whether two lists are **permutations of each other** — i.e. whether they contain exactly the same elements with exactly the same multiplicities, regardless of order.

```python
from collections import Counter
Counter([1, 2, 2]) == Counter([2, 1, 2])   # True
Counter([1, 2]) != Counter([1, 1])         # True
```

- `Counter` is the right tool because "is a permutation" is really **"sameness of the multiset"**: the same counts for the same elements.
- `Counter([1,2,2])` records `{1: 1, 2: 2}`; `Counter([2,1,2])` records the same, so the Counters compare equal — order never mattered.
- If lengths differ, they cannot be permutations — a cheap early exit before counting.
- Classic non-Counter alternative: `sorted(a) == sorted(b)` — identical once sorted; the Counter version is usually faster for large inputs because it avoids the O(n log n) sort.
- Sub-question interviewers love: when does `a == b` (order-sensitive) differ from "is permutation" (order-insensitive)? Any time the two lists hold the same elements in a different order — exactly the case the asserts show.


## Challenge 6

Check whether two lists are permutations of each other.

In [7]:
from collections import Counter
assert Counter([1, 2, 2]) == Counter([2, 1, 2])
assert Counter([1, 2]) != Counter([1, 1])
print('ok')

ok


### Challenge 7 — Chunk a dict into fixed-size pieces

The task: split a dictionary into a list of smaller dictionaries, each holding at most **n** consecutive items.

```python
d = {'a': 1, 'b': 2, 'c': 3}      # n = 2
items = list(d.items())
chunks = [dict(items[i:i + 2]) for i in range(0, len(items), 2)]
# len(chunks) == 2  ->  [{'a':1,'b':2}, {'c':3}]
```

- `d.items()` gives (key, value) pairs; `list(...)` snapshots them so slicing works.
- `range(0, len(items), 2)` steps `i` over the pair boundaries: 0, 2.
- `items[i:i + 2]` is the sliding slice of (at most) 2 pairs; `dict(...)` wraps the slice back into a dictionary.
- For a list of 3 items and size 2 you get 2 chunks — the second holds just `{'c': 3}`; the `len(chunks) == 2` assert checks that.
- The same slicing trick chunks *any* sequence — lists, tuples, strings — chained with whatever you need per piece; here the re-wrap with `dict()` is the data-type-specific step.


## Challenge 7

Chunk a dict into a list of smaller dicts of size n.

In [8]:
d = {'a': 1, 'b': 2, 'c': 3}
items = list(d.items())
chunks = [dict(items[i:i + 2]) for i in range(0, len(items), 2)]
assert len(chunks) == 2
print('ok')

ok


### Challenge 8 — Symmetric difference without `^`

The task: return the elements that are in **exactly one** of two sets — the symmetric difference — but *without* using the `^` operator (that is the exercise).

```python
def sym_diff(a, b):
    return (a - b) | (b - a)
```

- `a - b` is set difference: elements in `a` but not in `b`.
- `b - a` is the reverse: elements in `b` but not in `a`.
- The union `|` of the two differences contains every element that lives in exactly one set — by construction they overlap in nothing.
- `sym_diff({1,2}, {2,3})` → `({1} ) | ({3})` → `{1, 3}` — the assert.
- Sanity check against the operator form: `a ^ b` should equal the same set, and the natural test to convince yourself is `sym_diff(a, b) == (a ^ b)`.
- Why this exists as a challenge: it forces you to re-derive the definition instead of reaching for the built-in — the two-difference-then-union identity is a favourite "explain what `^` does" interview question.


## Challenge 8

Write a function returning the symmetric difference without using ^.

In [9]:
def sym_diff(a, b):
    return (a - b) | (b - a)

assert sym_diff({1, 2}, {2, 3}) == {1, 3}
print('ok')

ok


### Challenge 9 — Sort a list of dicts by two keys

The task: sort a list of dictionaries by **two criteria at once** — one field descending, another ascending. The example rows have `'n'` (name) and `'s'` (score); the expectation is highest score first, and for tied scores, alphabetical name.

```python
rows = [{'n': 'a', 's': 5}, {'n': 'b', 's': 5}, {'n': 'c', 's': 9}]
out = sorted(rows, key=lambda r: (-r['s'], r['n']))
```

- `sorted` accepts a `key` function that returns the *comparison value* of each row.
- The key here is a **tuple** `(-r['s'], r['n'])`. Tuples compare element by element: first by `-score`, then by name.
- Negating the score (`-r['s']`) flips its ordering — higher scores come first, which is the "descending" part without `reverse=True`.
- `'c'` (score 9) sorts before the two score-5 rows; among those two, `'a'` beats `'b'`, so `out[0]` is `{'n': 'c', 's': 9}` — the assert.
- This tuple-key + negation technique is the standard way to mix ascending and descending columns in a single `sorted` call; `reverse=True` would flip *both* fields, which is why negation is used for just one.


## Challenge 9

Sort a list of dicts by two keys, one descending.

In [10]:
rows = [{'n': 'a', 's': 5}, {'n': 'b', 's': 5}, {'n': 'c', 's': 9}]
out = sorted(rows, key=lambda r: (-r['s'], r['n']))
assert out[0]['n'] == 'c'
print('ok')

ok


### Challenge 10 — Build a nested dict from flat tuples

The task: convert a list of `(category, item, price)` tuples into a two-level dictionary — category maps to `{item: price}` — using one `setdefault` chain.

```python
rows = [('fruit', 'apple', 80), ('fruit', 'fig', 200), ('veg', 'okra', 40)]
out = {}
for cat, item, price in rows:
    out.setdefault(cat, {})[item] = price
```

- Tuple unpacking `for cat, item, price in rows:` reads the three fields of each row directly.
- `out.setdefault(cat, {})` returns the dict for `cat`, creating one if needed — then `[item] = price` plants the item→price pair inside it.
- After the loop: `out == {'fruit': {'apple': 80, 'fig': 200}, 'veg': {'okra': 40}}`; the assert `out['fruit']['fig'] == 200` drills two levels down.
- This is the **group-of-groups** pattern — exactly Challenge 1's grouping, but nested one level deeper (a dict *of* dicts rather than *of* lists).
- Structurally identical to building an index (`category → item → value`) and a close cousin of JSON-shaped data — the same idea underlies grouped reports and lookup tables in data work.


## Challenge 10

Build a nested dict from a list of (category, item, price) tuples.

In [11]:
rows = [('fruit', 'apple', 80), ('fruit', 'fig', 200), ('veg', 'okra', 40)]
out = {}
for cat, item, price in rows:
    out.setdefault(cat, {})[item] = price
assert out['fruit']['fig'] == 200
print('ok')

ok


## Summary

Across these ten challenges you exercised the core data-type patterns of the course. **Grouping**: words by first letter (Challenge 1) and flat rows into nested dicts (Challenge 10) via `setdefault`. **Set logic**: intersection with `&` over three lists (Challenge 2) and symmetric difference composed as `(a-b) | (b-a)` without `^` (Challenge 8). **Re-keying an inverted** mapping with `setdefault` (Challenge 3). **Frequency**: `Counter` for the k-most-frequent elements (Challenge 4) and for order-insensitive permutation checks (Challenge 6). **Merging with aggregation**: dict-union `|` plus `get(k, 0)` to sum shared keys (Challenge 5). **Chunking** any sequence with stepped slicing (Challenge 7). And **multi-key sorting** via a tuple key with negation for a single descending column (Challenge 9). Timed note from the closing card: ten minutes per challenge is a realistic screening pace — redo the ones that took longer. These patterns — grouping, intersection, counting, merging, chunking, multi-key sort — are the reusable building blocks behind most data-type interview questions.
